# Vessel-segmentation teacher for China-Fundus-CIMT pseudo-masks
Trains U-Net / U-Net++ / MAnet (ResNet-50 encoder) on DRIVE + STARE + FIVES, selects the best by validation Dice, evaluates on held-out test splits, and generates vessel masks for all 5,806 fundus images.

**Before running:** edit the paths in the config cell. Set `EPOCHS = 2` for a quick test first.

In [ ]:
!pip install -q segmentation-models-pytorch albumentations

## 1. Imports and config

In [ ]:
import os, glob, json, random, time
import numpy as np, cv2, torch, torch.nn as nn, torch.nn.functional as F
from PIL import Image
import albumentations as A
import segmentation_models_pytorch as smp
from sklearn.metrics import roc_auc_score

# ============================ PATHS / CONFIG ============================
DRIVE_ROOT = "/kaggle/input/drive"            # contains training/ and test/ (images, 1st_manual)
STARE_ROOT = "/kaggle/input/stare"            # contains images/*.ppm and labels-ah/*.ah.ppm
FIVES_ROOT = "/kaggle/input/fives"            # contains train/Original, train/Ground truth, test/...
CIMT_IMG_DIR = "/kaggle/input/china-fundus-cimt/images"   # PatientID_L.png / PatientID_R.png
OUT_DIR = "/kaggle/working/teacher"
SIZE = 512
ARCHS = [("Unet", "resnet50"), ("UnetPlusPlus", "resnet50"), ("MAnet", "resnet50")]
EPOCHS, BATCH, LR, SEED = 40, 8, 3e-4, 42
SAMPLES_PER_EPOCH = 480
DATASET_WEIGHT = {"FIVES": 0.5, "DRIVE": 0.25, "STARE": 0.25}   # sampling share per epoch
# ========================================================================
dev = "cuda" if torch.cuda.is_available() else "cpu"
os.makedirs(OUT_DIR, exist_ok=True)
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
MEAN = np.array([0.485, 0.456, 0.406], np.float32); STD = np.array([0.229, 0.224, 0.225], np.float32)


## 2. Preprocessing (resize, FOV, CLAHE)

In [ ]:
# ------------------------------ preprocessing ------------------------------
def read_rgb(p):  return np.array(Image.open(p).convert("RGB"))
def read_mask(p): return (np.array(Image.open(p).convert("L")) > 127).astype(np.uint8)

def get_fov(img):
    g = img.mean(2)
    m = (g > 12).astype(np.uint8)
    m = cv2.morphologyEx(m, cv2.MORPH_CLOSE, np.ones((15, 15), np.uint8))
    cnts, _ = cv2.findContours(m, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    out = np.zeros_like(m)
    if cnts: cv2.drawContours(out, [max(cnts, key=cv2.contourArea)], -1, 1, -1)
    else:    out[:] = 1
    return cv2.erode(out, np.ones((7, 7), np.uint8))

def clahe_rgb(img):
    lab = cv2.cvtColor(img, cv2.COLOR_RGB2LAB)
    lab[..., 0] = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8)).apply(lab[..., 0])
    return cv2.cvtColor(lab, cv2.COLOR_LAB2RGB)

def prep(img):
    """resize -> FOV -> CLAHE. returns uint8 image, uint8 fov"""
    img = cv2.resize(img, (SIZE, SIZE), interpolation=cv2.INTER_AREA)
    fov = get_fov(img)
    return clahe_rgb(img), fov

def to_tensor(img):
    x = (img.astype(np.float32) / 255. - MEAN) / STD
    return torch.from_numpy(x.transpose(2, 0, 1))


## 3. Dataset lists and splits

In [ ]:
# ------------------------------ data lists ------------------------------
def pair(img_glob, msk_glob):
    im, mk = sorted(glob.glob(img_glob)), sorted(glob.glob(msk_glob))
    assert len(im) == len(mk) > 0, f"pairing problem: {img_glob} ({len(im)}) vs {msk_glob} ({len(mk)})"
    return list(zip(im, mk))

def split(lst, n_val, n_test=0):
    r = random.Random(SEED); lst = lst[:]; r.shuffle(lst)
    return lst[n_val + n_test:], lst[:n_val], lst[n_val:n_val + n_test]

def build_lists():
    L = {}
    d_tr = pair(f"{DRIVE_ROOT}/training/images/*", f"{DRIVE_ROOT}/training/1st_manual/*")
    d_te = pair(f"{DRIVE_ROOT}/test/images/*", f"{DRIVE_ROOT}/test/1st_manual/*")
    tr, va, _ = split(d_tr, 4); L["DRIVE"] = dict(train=tr, val=va, test=d_te)
    s = pair(f"{STARE_ROOT}/images/*.ppm", f"{STARE_ROOT}/labels-ah/*.ppm")
    tr, va, te = split(s, 4, 4); L["STARE"] = dict(train=tr, val=va, test=te)
    f_tr = pair(f"{FIVES_ROOT}/train/Original/*", f"{FIVES_ROOT}/train/Ground truth/*")
    f_te = pair(f"{FIVES_ROOT}/test/Original/*", f"{FIVES_ROOT}/test/Ground truth/*")
    tr, va, _ = split(f_tr, 60); L["FIVES"] = dict(train=tr, val=va, test=f_te)
    return L

def load_cache(pairs):
    out = []
    for ip, mp in pairs:
        img, fov = prep(read_rgb(ip))
        m = cv2.resize(read_mask(mp), (SIZE, SIZE), interpolation=cv2.INTER_NEAREST)
        out.append((img, m, fov))
    return out


## 4. Augmentation and Dataset class

In [ ]:
aug = A.Compose([A.HorizontalFlip(), A.VerticalFlip(), A.Rotate(limit=30, p=0.7),
                 A.RandomBrightnessContrast(0.2, 0.2, p=0.5), A.RandomGamma(p=0.3)])

class DS(torch.utils.data.Dataset):
    def __init__(self, items, train):  # items: list of (img, mask, fov, name)
        self.items, self.train = items, train
    def __len__(self): return len(self.items)
    def __getitem__(self, i):
        img, m, fov, _ = self.items[i]
        if self.train:
            r = aug(image=img, masks=[m, fov]); img, (m, fov) = r["image"], r["masks"]
        return to_tensor(img), torch.from_numpy(m[None]).float(), torch.from_numpy(fov[None]).float()


## 5. Model, loss, TTA and metrics

In [ ]:
# ------------------------------ model / loss / metrics ------------------------------
def make_model(arch, enc):
    return getattr(smp, arch)(encoder_name=enc, encoder_weights="imagenet", in_channels=3, classes=1).to(dev)

def loss_fn(logits, y, fov):
    bce = (F.binary_cross_entropy_with_logits(logits, y, reduction="none") * fov).sum() / fov.sum().clamp(min=1)
    p = torch.sigmoid(logits) * fov; yy = y * fov
    inter = (p * yy).sum((1, 2, 3))
    dice = 1 - (2 * inter + 1) / (p.sum((1, 2, 3)) + yy.sum((1, 2, 3)) + 1)
    return bce + dice.mean()

@torch.no_grad()
def predict_prob(model, x):
    model.eval(); ps = 0
    for fl in [(), (3,), (2,), (2, 3)]:
        xi = torch.flip(x, fl) if fl else x
        with torch.autocast(dev, enabled=(dev == "cuda")):
            p = torch.sigmoid(model(xi).float())
        ps = ps + (torch.flip(p, fl) if fl else p)
    return ps / 4

def metrics(prob, m, fov, thr):
    """per-image Dice/IoU/SE/SP inside FOV; prob,m,fov are numpy HxW"""
    pb = (prob > thr) & (fov > 0); g = (m > 0) & (fov > 0); f = fov > 0
    tp = (pb & g).sum(); fp = (pb & ~g & f).sum(); fn = (~pb & g).sum(); tn = (~pb & ~g & f).sum()
    return dict(dice=2 * tp / (2 * tp + fp + fn + 1e-8), iou=tp / (tp + fp + fn + 1e-8),
                se=tp / (tp + fn + 1e-8), sp=tn / (tn + fp + 1e-8))

def eval_set(model, items, thr=0.5, auc=False):
    probs, rows, ys, ps = [], [], [], []
    for img, m, fov, _ in items:
        x = to_tensor(img)[None].to(dev)
        p = predict_prob(model, x)[0, 0].cpu().numpy(); probs.append(p)
        rows.append(metrics(p, m, fov, thr))
        if auc: idx = np.flatnonzero(fov.ravel() > 0); ys.append(m.ravel()[idx]); ps.append(p.ravel()[idx])
    out = {k: float(np.mean([r[k] for r in rows])) for k in rows[0]}
    if auc: out["auc"] = float(roc_auc_score(np.concatenate(ys), np.concatenate(ps)))
    return out, probs

def train_one(arch, enc, tr_items, tr_w, val_sets):
    model = make_model(arch, enc)
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
    steps = EPOCHS * (SAMPLES_PER_EPOCH // BATCH)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=LR, total_steps=steps, pct_start=0.1)
    scaler = torch.cuda.amp.GradScaler(enabled=(dev == "cuda"))
    sampler = torch.utils.data.WeightedRandomSampler(tr_w, SAMPLES_PER_EPOCH, replacement=True)
    dl = torch.utils.data.DataLoader(DS(tr_items, True), BATCH, sampler=sampler, num_workers=2, drop_last=True)
    best, best_state = -1, None
    for ep in range(EPOCHS):
        model.train(); t0 = time.time(); tl = 0
        for x, y, fov in dl:
            x, y, fov = x.to(dev), y.to(dev), fov.to(dev)
            with torch.autocast(dev, enabled=(dev == "cuda")):
                lg = model(x)
            loss = loss_fn(lg.float(), y, fov)
            opt.zero_grad(set_to_none=True); scaler.scale(loss).backward()
            scaler.step(opt); scaler.update(); sched.step(); tl += loss.item()
        if ep % 2 == 1 or ep == EPOCHS - 1:
            v = np.mean([eval_set(model, it)[0]["dice"] for it in val_sets.values()])  # macro over datasets
            if v > best: best, best_state = v, {k: t.clone() for k, t in model.state_dict().items()}
            print(f"[{arch}-{enc}] ep{ep+1:02d} loss {tl/len(dl):.4f} valDice {v:.4f} best {best:.4f} ({time.time()-t0:.0f}s)")
    model.load_state_dict(best_state)
    return model, best


## 6. Load data into memory

In [ ]:
L = build_lists()
data = {n: {s: [(*c, os.path.basename(p[0])) for c, p in zip(load_cache(L[n][s]), L[n][s])]
            for s in ("train", "val", "test")} for n in L}
tr_items, tr_w = [], []
for n in L:
    for it in data[n]["train"]:
        tr_items.append(it); tr_w.append(DATASET_WEIGHT[n] / len(data[n]["train"]))
val_sets = {n: data[n]["val"] for n in L}
print({n: {s: len(v) for s, v in d.items()} for n, d in data.items()})


## 7. Train candidate architectures and select the best

In [ ]:
results, best_model, best_name, best_val = {}, None, None, -1
for arch, enc in ARCHS:
    m, v = train_one(arch, enc, tr_items, tr_w, val_sets); results[f"{arch}-{enc}"] = v
    if v > best_val: best_model, best_name, best_val = m, f"{arch}-{enc}", v
print("validation Dice per architecture:", results, "-> best:", best_name)


## 8. Tune threshold, held-out evaluation, save model and report

In [ ]:
# threshold tuning on validation (macro over datasets)
val_probs = {n: eval_set(best_model, val_sets[n])[1] for n in L}
grid = np.arange(0.30, 0.71, 0.05)
score = [np.mean([np.mean([metrics(p, m, f, t)["dice"] for p, (_, m, f, _) in zip(val_probs[n], val_sets[n])])
                  for n in L]) for t in grid]
thr = float(grid[int(np.argmax(score))]); print("threshold:", thr)

report = dict(best=best_name, val_dice_by_arch=results, threshold=thr, heldout={})
for n in L:   # held-out test splits (never used for selection)
    report["heldout"][n], _ = eval_set(best_model, data[n]["test"], thr, auc=True)
print(json.dumps(report, indent=2))
json.dump(report, open(f"{OUT_DIR}/teacher_report.json", "w"), indent=2)
torch.save(best_model.state_dict(), f"{OUT_DIR}/teacher_{best_name}.pt")


## 9. Generate pseudo-masks for all China-Fundus-CIMT images

In [ ]:
# ---------- pseudo-label all China-Fundus-CIMT images ----------
os.makedirs(f"{OUT_DIR}/masks", exist_ok=True); os.makedirs(f"{OUT_DIR}/overlays", exist_ok=True)
files = sorted(glob.glob(f"{CIMT_IMG_DIR}/*.png")); dens = []
for i, fp in enumerate(files):
    raw = read_rgb(fp); img, fov = prep(raw)
    p = predict_prob(best_model, to_tensor(img)[None].to(dev))[0, 0].cpu().numpy()
    mask = ((p > thr) & (fov > 0)).astype(np.uint8)
    name = os.path.basename(fp)
    Image.fromarray(mask * 255).save(f"{OUT_DIR}/masks/{name}")
    dens.append(mask.sum() / max(1, fov.sum()))
    if i % 250 == 0:   # visual QC overlays
        ov = cv2.resize(raw, (SIZE, SIZE)).copy(); ov[mask > 0] = (0, 255, 0)
        Image.fromarray(np.hstack([cv2.resize(raw, (SIZE, SIZE)), ov])).save(f"{OUT_DIR}/overlays/{name}")
print(f"{len(files)} masks saved. vessel density in FOV: mean {np.mean(dens):.3f}, "
      f"min {np.min(dens):.3f}, max {np.max(dens):.3f}")


## 10. Visual QC and report

In [ ]:
import matplotlib.pyplot as plt
ov = sorted(glob.glob(f"{OUT_DIR}/overlays/*.png"))[:6]
fig, ax = plt.subplots(len(ov), 1, figsize=(10, 5*len(ov)))
ax = np.atleast_1d(ax)
for a, p in zip(ax, ov):
    a.imshow(Image.open(p)); a.set_title(os.path.basename(p)); a.axis("off")
plt.tight_layout(); plt.show()
print(open(f"{OUT_DIR}/teacher_report.json").read())
